In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams.update({
    'font.size': 15, 'lines.linewidth': 2,
    'xtick.labelsize': 13, 'ytick.labelsize': 13,
    'axes.spines.top': False, 'axes.spines.right': False,
    'savefig.dpi': 1200,
})

import yaml
import numpy as np

import sys
sys.path.append('/Users/lokeshboominathan/OneDrive - Rice University/Projects/Auditory-foraging-IRC/AF_setup_v3/irc_gym')

from irc.manager import IRCManager
from auditoryforage.utils import plot_auditory_foraging_v1_episode

# Train one agent for a single environment

In [ ]:
defaults = {'agent.env._target_': 'auditoryforage.AF_env.AuditoryForaging'}
manager = IRCManager(defaults=defaults)

## Train an agent
We train a rational agent for the assume environment parameter $lick\_cost=-3.0, food\_reward=12.0, high\_attention\_cost=-3.0$.

The following can also be done by running `demo-train.py` in command line:
```bash
python demo-train.py env_param=[-3.0,12.0,-3.0] num_epochs=20
```

In [ ]:
env_param = [-3.0, 12.0, -3.0]
num_epochs = 20


agent = manager.train_agent(env_param, num_epochs=num_epochs)
agent, fig = manager.inspect_agent(env_param)

## Run the agent in an environment
We create another environment which has the same observation space and action space as the assumed one, albeit with a different set of environment parameters $(-3.0, 12.0, -3.0)$.

In [ ]:
from auditoryforage.AF_env import AuditoryForaging

env = AuditoryForaging(spec={'experiment': {'prob_01': 0.5},'agent':{'lick_cost':-3,'food_reward':12,'high_attention_cost':-3}})
env.spec

episode = agent.run_one_episode(env=env, num_steps=60, q_states = [[i] for i in range(7)])
fig = plot_auditory_foraging_v1_episode(episode)

We can save the episode data in an external `pickle` file.

In [ ]:
import os, pickle

episode_path = 'store/episode_00.pickle'

if os.path.exists(episode_path):
    print(f"File {episode_path} already exists, will not be overwritten.")
else:
    with open(episode_path, 'wb') as f:
        pickle.dump({
            'external_env_param': env.get_param(),
            'internal_env_param': agent.model.env.get_param(),
            **episode,
        }, f)
    print(f"Episode data saved at '{episode_path}'.")

## Sanity check

We show that belief updates done using anayltical update (from environemt file) is the same as what we get using sampling approach (previous plot)$.

In [ ]:
# actions = episode["actions"]
# observations = episode["observations"]
# offset_actions = np.insert(actions, -1, 0, axis=0) #offset actions to match time steps, as the action at last time step is not taken.
# previous_belief = 1/7 * np.ones((7,1))
# computed_beliefs = []
# for ind in range(len(offset_actions)):
#     try:
#         next_belief = env.update_belief(previous_belief, observations[ind][0], offset_actions[ind-1])
#     except:
#         next_belief = env.update_belief(previous_belief, observations[ind][0], 0)
#     computed_beliefs.append(next_belief)
#     previous_belief = next_belief
# episode["q_probs"] = np.array(computed_beliefs)
# fig = plot_auditory_foraging_v1_episode(episode)

# Train multiple agents for different environments

## Sweep over parameter grid

We define a grid of environment parameters, and train multiple agents for each of the combination using different random seeds.

The following can also be done by running `demo-sweep.py` in command line:
```bash
python demo-sweep.py env_param_grid=param_grids/param_grid_0.yaml num_epochs=20 count=5
```

In [ ]:
# Lokesh changed to smaller search space.

env_param_grid = [
    [-3.0], # lick_cost
    [1.0, 8.0, 12.0, 50.0], # food_reward
    [-3.0], # high_attention_cost
]

num_epochs = 20 # number of RL epochs

# manager.train_agents(env_param_grid=env_param_grid, num_epochs=num_epochs, count=count)
manager.train_agents(env_param_grid=env_param_grid, num_epochs=num_epochs)

In [ ]:
report = manager.agents_overview(env_param_grid=env_param_grid)

# Compute likelihood of an episode

## Likelihood of a specific agent

Given a sequence actions and observations, we compute the likelihood of episode data conditioned on a trained agent $p(a_{1:t}, o_{1:t}|\theta_\mathrm{agent})$.

In [ ]:
logp = agent.episode_likelihood(episode['actions'], episode['observations'])
print('log likelihood {:.3f}'.format(logp))

## Likelihood of a series of agents

After multiple agents have been trained on a grid of environment parameters, compute the likelihood of episode data conditioned on each agent.

In [ ]:
from irc.utils import logmeanexp

episode_path = 'episode_00.pickle'

# Lokesh changed to smaller search space.
env_param_grid = [
    [-3.0], # lick_cost
    [1.0, 8.0, 12.0, 50.0], # food_reward
    [-3.0], # high_attention_cost
]

logps = manager.compute_logps(episode_path=episode_path, env_param_grid=env_param_grid, min_epoch=5)

logps = logmeanexp(logps, axis=(-2, -1))
print(f"Log likelihood of all environments saved in an array of shape {logps.shape}")